# Assignment Beta - Question 1

**Name:** Muhammad Sabeeh  
**Roll number:** 23K-0002  
**Course:** Agentic AI (AI4015)  
**Semester:** Fall 2026


# Assignment Beta - Question 1
Muhammad Sabeeh | 23K-0002; R=2, D=7. Open-model prices were retrieved from live public endpoint listings. Latency speeds and cache-hit behavior are planning assumptions. No missing figure is represented by zero.

In [1]:
import math, statistics, json
from pathlib import Path
ROOT=Path.cwd()
ROLL="23K-0002"
def q1():
    digits=[int(c) for c in ROLL if c.isdigit()]; R=10*digits[-2]+digits[-1]; D=sum(digits)%10
    Q=1500+25*R; N=5+R%4; p=round(.04+.01*D,2); H=4+R%4; B=1200+10*R; T=10-R%3
    steps=[{'k':k,'input':2000+800*(k-1),'output':600 if k==N else 350,
            'attempts_expected':1 if k==N else 1+p} for k in range(1,N+1)]
    I=round(sum(s['input']*s['attempts_expected'] for s in steps),6)
    O=round(sum(s['output']*s['attempts_expected'] for s in steps),6)
    C=round(sum(s['attempts_expected'] for s in steps),6); M=Q*(30+4*H)
    peak_qpm=5*Q*.35/120
    cfgpath=ROOT/'sources'/'qwen3_32b_config.json'
    cfg=json.loads(cfgpath.read_text()) if cfgpath.exists() else {'num_hidden_layers':64,'num_key_value_heads':8,'hidden_size':5120,'num_attention_heads':64,'head_dim':128}
    layers=cfg['num_hidden_layers']; kv=cfg['num_key_value_heads']; dim=cfg.get('head_dim',cfg['hidden_size']//cfg['num_attention_heads'])
    params=32762123264  # BF16 safetensors metadata 65,524,246,528 bytes / 2.
    index=ROOT/'sources'/'qwen3_32b_metadata.json'
    if index.exists():
        data=json.loads(index.read_text())
        params=data['metadata']['total_size']//2
    seq=steps[-1]['input']+steps[-1]['output']
    cache_bytes=2*layers*kv*dim*seq*2
    # A100 80GB treated conservatively as 80e9 bytes, not 80GiB.
    capacity=math.floor((.9*80e9-params/2)/cache_bytes)
    baseline_latency_seconds=60.0  # Capacity scenario; never a measured serving latency.
    sessions=math.ceil(peak_qpm*baseline_latency_seconds/60)
    gpus=math.ceil(sessions/capacity)
    return {'student_roll':ROLL,'student_name':'Muhammad Sabeeh','R':R,'D':D,'Q':Q,'N':N,'p':p,'H':H,'B':B,'T':T,'steps':steps,
            'input_per_query':I,'output_per_query':O,'calls_per_query':C,'monthly_queries':M,
            'monthly_input':M*I,'monthly_output':M*O,'monthly_calls':M*C,
            'prefix_per_query':1500*C,'peak_queries_per_minute':peak_qpm,
            'peak_calls_per_minute':peak_qpm*C,'peak_input_tpm':peak_qpm*I,
            'peak_output_tpm':peak_qpm*O,'peak_total_tpm':peak_qpm*(I+O),
            'memory':{'parameter_count':params,'layers':layers,'kv_heads':kv,'head_dim':dim,'sequence_length':seq,
                      'fp16_weight_bytes':params*2,'int8_weight_bytes':params,'int4_weight_bytes':params/2,
                      'kv_bytes_per_sequence':cache_bytes,'gpu_bytes':80e9,'concurrent_sequences_per_gpu':capacity,
                      'assumed_session_seconds':baseline_latency_seconds,'peak_sessions':sessions,'gpu_count':gpus,
                      'gpu_usd_per_hour':1.59,'monthly_usd':gpus*720*1.59}}


In [2]:
x=q1()
print(json.dumps(x,indent=2))

{
  "student_roll": "23K-0002",
  "student_name": "Muhammad Sabeeh",
  "R": 2,
  "D": 7,
  "Q": 1550,
  "N": 7,
  "p": 0.11,
  "H": 6,
  "B": 1220,
  "T": 8,
  "steps": [
    {
      "k": 1,
      "input": 2000,
      "output": 350,
      "attempts_expected": 1.11
    },
    {
      "k": 2,
      "input": 2800,
      "output": 350,
      "attempts_expected": 1.11
    },
    {
      "k": 3,
      "input": 3600,
      "output": 350,
      "attempts_expected": 1.11
    },
    {
      "k": 4,
      "input": 4400,
      "output": 350,
      "attempts_expected": 1.11
    },
    {
      "k": 5,
      "input": 5200,
      "output": 350,
      "attempts_expected": 1.11
    },
    {
      "k": 6,
      "input": 6000,
      "output": 350,
      "attempts_expected": 1.11
    },
    {
      "k": 7,
      "input": 6800,
      "output": 600,
      "attempts_expected": 1
    }
  ],
  "input_per_query": 33440.0,
  "output_per_query": 2931.0,
  "calls_per_query": 7.66,
  "monthly_queries": 83700,
  "mon

## Twelve-model cost register
Open-model prices are the cheapest observed active endpoint for this workload among the retrieved listings, including flex tiers. They do not establish a global market minimum. The cache scenario assumes 720 creations per month and warm-prefix hits otherwise. New paid rate limits, TTFT and several benchmark scores remain unresolved.

In [3]:
MODELS=[{'model': 'GPT-4.1 mini', 'side': 'closed', 'developer': 'OpenAI', 'parameters': 'not disclosed', 'context_tokens': 1047576, 'max_output_tokens': 32768, 'input_usd_per_million': 0.4, 'output_usd_per_million': 1.6, 'cache_read_usd_per_million': 0.1, 'cache_minimum_tokens': 1024, 'cache_ttl': '5-10 min inactivity, up to 1 h', 'cache_write_multiplier': 1, 'rate_provider': 'OpenAI', 'new_paid_rpm': 'NR: account/tier-specific', 'new_paid_tpm': 'NR: account/tier-specific', 'ttft_seconds': None, 'output_tokens_per_second': None, 'tool_benchmark': 'BFCL v4 FC: 50.45% [B1]', 'general_benchmark': 'MMLU: 87.5%; OpenAI launch table [B2]', 'native_tools': 'Yes', 'structured_json': 'Yes', 'reasoning_setting': 'Non-reasoning; no reasoning-effort mode', 'license': 'Proprietary', 'commercial_self_host': 'No', 'source_url': 'https://developers.openai.com/api/docs/models/gpt-4.1-mini', 'access_date': '2026-11-25', 'evidence_status': 'Retrieved official context/output/prices; rate limits, speed and benchmark scores NR'}, {'model': 'GPT-4.1 nano', 'side': 'closed', 'developer': 'OpenAI', 'parameters': 'not disclosed', 'context_tokens': 1047576, 'max_output_tokens': 32768, 'input_usd_per_million': 0.1, 'output_usd_per_million': 0.4, 'cache_read_usd_per_million': 0.025, 'cache_minimum_tokens': 1024, 'cache_ttl': '5-10 min inactivity, up to 1 h', 'cache_write_multiplier': 1, 'rate_provider': 'OpenAI', 'new_paid_rpm': 'NR: account/tier-specific', 'new_paid_tpm': 'NR: account/tier-specific', 'ttft_seconds': 0.87, 'output_tokens_per_second': 174, 'tool_benchmark': 'BFCL v4 FC (2025-04-14): 33.05% [B1]', 'general_benchmark': 'MMLU: 80.1% [B2]', 'native_tools': 'Yes', 'structured_json': 'Yes', 'reasoning_setting': 'Non-reasoning; no reasoning-effort mode', 'license': 'Proprietary', 'commercial_self_host': 'No', 'source_url': 'https://developers.openai.com/api/docs/models/gpt-4.1-nano', 'access_date': '2026-11-25', 'evidence_status': 'Retrieved official model listing; price supported by GPT-4.1 launch table; remaining technical fields need recheck', 'speed_source': 'https://artificialanalysis.ai/models/gpt-4-1-nano', 'speed_caveat': 'Reported profile is non-reasoning; page lists Azure and flags deprecated. Not a measured OpenAI-direct latency at Karachi or across every context length. The headline output-rate definition may include end-to-end time: it is not substituted for an independently measured decode rate in latency arithmetic.'}, {'model': 'Claude Haiku 4.5', 'side': 'closed', 'developer': 'Anthropic', 'parameters': 'not disclosed', 'context_tokens': 200000, 'max_output_tokens': 64000, 'input_usd_per_million': 1, 'output_usd_per_million': 5, 'cache_read_usd_per_million': 0.1, 'cache_minimum_tokens': 4096, 'cache_ttl': '5 min default; 1 h optional; cache write multipliers 1.25x / 2x', 'cache_write_multiplier': 1.25, 'rate_provider': 'Anthropic', 'new_paid_rpm': 'NR: account/tier-specific', 'new_paid_tpm': 'NR: account/tier-specific', 'ttft_seconds': None, 'output_tokens_per_second': None, 'tool_benchmark': 'BFCL v4 FC: 68.70% [B1]', 'general_benchmark': 'SWE-bench Verified: 73.3%; 50 trials, 128K thinking budget; not non-thinking evidence', 'native_tools': 'Yes', 'structured_json': 'Yes', 'reasoning_setting': 'thinking disabled; optional extended thinking', 'license': 'Proprietary', 'commercial_self_host': 'No', 'source_url': 'https://platform.claude.com/docs/en/about-claude/pricing', 'access_date': '2026-11-25', 'evidence_status': 'Retrieved official prices; other fields are documented configurations awaiting source recapture', 'cache_policy_source': 'https://platform.claude.com/docs/en/build-with-claude/prompt-caching', 'benchmark_source': 'https://www.anthropic.com/news/claude-haiku-4-5'}, {'model': 'Claude Sonnet 4.5', 'side': 'closed', 'developer': 'Anthropic', 'parameters': 'not disclosed', 'context_tokens': 200000, 'max_output_tokens': 64000, 'input_usd_per_million': 3, 'output_usd_per_million': 15, 'cache_read_usd_per_million': 0.3, 'cache_minimum_tokens': 1024, 'cache_ttl': '5 min default; 1 h optional; cache write multipliers 1.25x / 2x', 'cache_write_multiplier': 1.25, 'rate_provider': 'Anthropic', 'new_paid_rpm': 'NR: account/tier-specific', 'new_paid_tpm': 'NR: account/tier-specific', 'ttft_seconds': None, 'output_tokens_per_second': None, 'tool_benchmark': 'BFCL v4 FC (20250929): 73.24% [B1]', 'general_benchmark': 'SWE-bench Verified: 77.2%; 10 trials, 200K thinking budget; not non-thinking evidence', 'native_tools': 'Yes', 'structured_json': 'Yes', 'reasoning_setting': 'thinking disabled; optional extended thinking', 'license': 'Proprietary', 'commercial_self_host': 'No', 'source_url': 'https://platform.claude.com/docs/en/about-claude/pricing', 'access_date': '2026-11-25', 'evidence_status': 'Retrieved official prices; other fields are documented configurations awaiting source recapture', 'cache_policy_source': 'https://platform.claude.com/docs/en/build-with-claude/prompt-caching', 'benchmark_source': 'https://www.anthropic.com/news/claude-sonnet-4-5'}, {'model': 'Gemini 2.5 Flash-Lite', 'side': 'closed', 'developer': 'Google', 'parameters': 'not disclosed', 'context_tokens': 1048576, 'max_output_tokens': 65536, 'input_usd_per_million': 0.1, 'output_usd_per_million': 0.4, 'cache_read_usd_per_million': 0.01, 'cache_minimum_tokens': 1024, 'cache_ttl': 'Explicit default 1 h; configurable', 'cache_write_multiplier': 1, 'rate_provider': 'Google', 'new_paid_rpm': 'NR: account/tier-specific', 'new_paid_tpm': 'NR: account/tier-specific', 'ttft_seconds': 0.35, 'output_tokens_per_second': None, 'tool_benchmark': 'BFCL v4 FC: 36.87% [B1]', 'general_benchmark': 'Global MMLU Lite: 81.1%, June 17 non-thinking release', 'native_tools': 'Yes', 'structured_json': 'Yes', 'reasoning_setting': 'thinking_budget=0; optional thinking budget', 'license': 'Proprietary', 'commercial_self_host': 'No', 'source_url': 'https://ai.google.dev/gemini-api/docs/pricing', 'access_date': '2026-11-25', 'evidence_status': 'Retrieved public listing and pricing search result; caching/limits need recapture', 'benchmark_source': 'https://storage.googleapis.com/deepmind-media/Model-Cards/Gemini-2-5-Flash-Lite-Model-Card.pdf', 'speed_source': 'https://artificialanalysis.ai/models/gemini-2-5-flash-lite/providers', 'speed_caveat': 'Non-reasoning provider table reports .35 s, FAQ .36 s; no decode-rate value retrieved. Do not reuse reasoning-mode 10.92 s.'}, {'model': 'Grok 4.7', 'side': 'closed', 'developer': 'xAI', 'parameters': 'not disclosed', 'context_tokens': 500000, 'max_output_tokens': None, 'input_usd_per_million': 2, 'output_usd_per_million': 6, 'cache_read_usd_per_million': 0.5, 'cache_minimum_tokens': None, 'cache_ttl': 'NR', 'cache_write_multiplier': 1, 'rate_provider': 'xAI', 'new_paid_rpm': 'Public model ceiling 150 RPS = 9000 RPM; new-account entitlement unverified', 'new_paid_tpm': 'Public model ceiling 50,000,000 TPM; new-account entitlement unverified', 'ttft_seconds': None, 'output_tokens_per_second': None, 'tool_benchmark': 'Terminal-Bench 4.0: 37.6%, xhigh; agentic terminal-use benchmark; not BFCL', 'general_benchmark': 'EEBench: 64.0%, xhigh; not low-effort evidence', 'native_tools': 'Yes', 'structured_json': 'Yes', 'reasoning_setting': 'reasoning_effort=low; supports low/medium/high/xhigh; residual reasoning volume unknown', 'license': 'Proprietary', 'commercial_self_host': 'No', 'source_url': 'https://docs.x.ai/developers/models/grok-4.7', 'access_date': '2026-11-25', 'evidence_status': 'Retrieved official context/input/output prices; total cost is a lower bound until reasoning tokens measured', 'benchmark_source': 'https://x.ai/news/grok-4-7'}, {'model': 'Qwen3-235B-A22B-Instruct-2507', 'side': 'open', 'developer': 'Alibaba Qwen', 'parameters': '235B total / 22B active MoE', 'context_tokens': 262144, 'max_output_tokens': 235929, 'input_usd_per_million': 0.072, 'output_usd_per_million': 0.44, 'cache_read_usd_per_million': None, 'cache_minimum_tokens': None, 'cache_ttl': 'NR', 'cache_write_multiplier': 1, 'rate_provider': 'DeepInfra / deepinfra/fp8/flex', 'new_paid_rpm': 'RPM not published; 200 concurrent requests/model on direct DeepInfra', 'new_paid_tpm': 'TPM not published; direct-provider concurrency is not a router entitlement', 'ttft_seconds': None, 'output_tokens_per_second': None, 'tool_benchmark': 'BFCL v3: 70.9% [B5]', 'general_benchmark': 'MMLU-Redux: 93.1% [B5]', 'native_tools': 'Yes; endpoint tool-choice modes vary', 'structured_json': 'Yes; provider structured output', 'reasoning_setting': 'Non-thinking instruct', 'license': 'Apache-2.0', 'commercial_self_host': 'Yes', 'source_url': 'https://openrouter.ai/api/v1/models/qwen/qwen3-235b-a22b-2507/endpoints', 'access_date': '2026-11-25', 'evidence_status': 'Retrieved public endpoint prices/context/output/features; cache policy/limits/speed NR', 'quota_source': 'https://docs.deepinfra.com/account/rate-limits'}, {'model': 'Qwen3-32B', 'side': 'open', 'developer': 'Alibaba Qwen', 'parameters': '32.762B dense', 'context_tokens': 40960, 'max_output_tokens': 16384, 'input_usd_per_million': 0.064, 'output_usd_per_million': 0.224, 'cache_read_usd_per_million': None, 'cache_minimum_tokens': None, 'cache_ttl': 'NR', 'cache_write_multiplier': 1, 'rate_provider': 'DeepInfra / deepinfra/fp8/flex', 'new_paid_rpm': 'RPM not published; 200 concurrent requests/model on direct DeepInfra', 'new_paid_tpm': 'TPM not published; direct DeepInfra quota is concurrency, not tokens/minute', 'ttft_seconds': None, 'output_tokens_per_second': None, 'tool_benchmark': 'BFCL v3: 63.0%, non-thinking, Table 14', 'general_benchmark': 'MMLU-Redux: 85.7%, non-thinking, Table 14', 'native_tools': 'Yes; endpoint tool-choice modes vary', 'structured_json': 'Yes; provider structured output', 'reasoning_setting': 'enable_thinking=False', 'license': 'Apache-2.0', 'commercial_self_host': 'Yes', 'source_url': 'https://openrouter.ai/api/v1/models/qwen/qwen3-32b/endpoints', 'access_date': '2026-11-25', 'evidence_status': 'Retrieved public endpoint prices/context/output/features; cache policy/limits/speed NR', 'quota_source': 'https://docs.deepinfra.com/account/rate-limits', 'benchmark_source': 'https://arxiv.org/html/2505.09388v1'}, {'model': 'Llama-3.3-70B-Instruct', 'side': 'open', 'developer': 'Meta', 'parameters': '70B dense', 'context_tokens': 131072, 'max_output_tokens': 16384, 'input_usd_per_million': 0.09999999999999999, 'output_usd_per_million': 0.32, 'cache_read_usd_per_million': None, 'cache_minimum_tokens': None, 'cache_ttl': 'NR', 'cache_write_multiplier': 1, 'rate_provider': 'DeepInfra / deepinfra/turbo', 'new_paid_rpm': 'RPM not published; 200 concurrent requests/model on direct DeepInfra', 'new_paid_tpm': 'TPM not published; concurrency policy is not an RPM/TPM guarantee', 'ttft_seconds': None, 'output_tokens_per_second': None, 'tool_benchmark': 'BFCL v2: 77.3% [B3]', 'general_benchmark': 'MMLU Pro CoT 5-shot: 68.9% [B3]', 'native_tools': 'Yes; endpoint tool-choice modes vary', 'structured_json': 'Yes; provider structured output', 'reasoning_setting': 'Non-reasoning instruct', 'license': 'Llama 3.3 Community', 'commercial_self_host': 'Conditional; review licence', 'source_url': 'https://openrouter.ai/api/v1/models/meta-llama/llama-3.3-70b-instruct/endpoints', 'access_date': '2026-11-25', 'evidence_status': 'Retrieved public endpoint prices/context/output/features; cache policy/limits/speed NR', 'quota_source': 'https://docs.deepinfra.com/account/rate-limits'}, {'model': 'Mistral-Small-3.2-24B-Instruct', 'side': 'open', 'developer': 'Mistral AI', 'parameters': '24B dense', 'context_tokens': 128000, 'max_output_tokens': 102400, 'input_usd_per_million': 0.06, 'output_usd_per_million': 0.16, 'cache_read_usd_per_million': None, 'cache_minimum_tokens': None, 'cache_ttl': 'NR', 'cache_write_multiplier': 1, 'rate_provider': 'DeepInfra / deepinfra/fp8/flex', 'new_paid_rpm': 'RPM not published; 200 concurrent requests/model on direct DeepInfra', 'new_paid_tpm': 'TPM not published; concurrency policy is not an RPM/TPM guarantee', 'ttft_seconds': None, 'output_tokens_per_second': None, 'tool_benchmark': 'BFCL v4 FC (Mistral-small-2506): 37.15% [B1]', 'general_benchmark': 'MMLU: 80.50% [B4]', 'native_tools': 'Yes; endpoint tool-choice modes vary', 'structured_json': 'Yes; provider structured output', 'reasoning_setting': 'Non-reasoning instruct', 'license': 'Apache-2.0', 'commercial_self_host': 'Yes', 'source_url': 'https://openrouter.ai/api/v1/models/mistralai/mistral-small-3.2-24b-instruct/endpoints', 'access_date': '2026-11-25', 'evidence_status': 'Retrieved public endpoint prices/context/output/features; cache policy/limits/speed NR', 'quota_source': 'https://docs.deepinfra.com/account/rate-limits'}, {'model': 'DeepSeek-V3.1', 'side': 'open', 'developer': 'DeepSeek', 'parameters': '671B total / 37B active MoE', 'context_tokens': 163840, 'max_output_tokens': 147456, 'input_usd_per_million': 0.25, 'output_usd_per_million': 0.95, 'cache_read_usd_per_million': 0.13, 'cache_minimum_tokens': None, 'cache_ttl': 'NR', 'cache_write_multiplier': 1, 'rate_provider': 'DeepInfra / deepinfra/fp4', 'new_paid_rpm': 'RPM not published; 200 concurrent requests/model on direct DeepInfra', 'new_paid_tpm': 'TPM not published; concurrency policy is not an RPM/TPM guarantee', 'ttft_seconds': None, 'output_tokens_per_second': None, 'tool_benchmark': 'Terminal-bench / Terminus 1: 31.3%, non-thinking', 'general_benchmark': 'MMLU-Redux: 91.8%, non-thinking', 'native_tools': 'Yes; endpoint tool-choice modes vary', 'structured_json': 'Yes; provider structured output', 'reasoning_setting': 'Non-thinking template', 'license': 'MIT', 'commercial_self_host': 'Yes', 'source_url': 'https://openrouter.ai/api/v1/models/deepseek/deepseek-chat-v3.1/endpoints', 'access_date': '2026-11-25', 'evidence_status': 'Retrieved public endpoint prices/context/output/features; cache policy/limits/speed NR', 'quota_source': 'https://docs.deepinfra.com/account/rate-limits', 'benchmark_source': 'https://huggingface.co/deepseek-ai/DeepSeek-V3.1/raw/main/README.md'}, {'model': 'gpt-oss-120b', 'side': 'open', 'developer': 'OpenAI', 'parameters': '117B total / 5.1B active MoE', 'context_tokens': 131072, 'max_output_tokens': 117964, 'input_usd_per_million': 0.03, 'output_usd_per_million': 0.15, 'cache_read_usd_per_million': 0.015, 'cache_minimum_tokens': None, 'cache_ttl': 'NR', 'cache_write_multiplier': 1, 'rate_provider': 'Near AI / near-ai/fp4', 'new_paid_rpm': 'NR: account/tier-specific', 'new_paid_tpm': 'NR: account/tier-specific', 'ttft_seconds': None, 'output_tokens_per_second': None, 'tool_benchmark': 'Tau-Bench Retail: 67.8%, high reasoning', 'general_benchmark': 'GPQA Diamond (no tools): 80.1%, high reasoning', 'native_tools': 'Yes; endpoint tool-choice modes vary', 'structured_json': 'Yes; provider structured output', 'reasoning_setting': 'reasoning_effort=low; residual reasoning volume unknown', 'license': 'Apache-2.0', 'commercial_self_host': 'Yes', 'source_url': 'https://openrouter.ai/api/v1/models/openai/gpt-oss-120b/endpoints', 'access_date': '2026-11-25', 'evidence_status': 'Retrieved public endpoint prices/context/output/features; cache policy/limits/speed NR', 'benchmark_source': 'https://catalog.ngc.nvidia.com/orgs/nim/openai/models/gpt-oss-120b/-', 'benchmark_caveat': 'Both reported scores use high reasoning; neither establishes low-effort deployment performance.'}]

def model_cost(m,x):
    a=m['input_usd_per_million'];b=m['output_usd_per_million'];c=m['cache_read_usd_per_million']
    raw=(x['monthly_input']*a+x['monthly_output']*b)/1e6
    attempts=x['monthly_calls'];prefix=1500
    eligible=c is not None and m['cache_minimum_tokens'] is not None and prefix>=m['cache_minimum_tokens']
    # One cache creation per hour as a conservative explicit refresh policy.
    # All other matching calls hit the cache. Assumes traffic is regular enough
    # to keep the prefix warm; actual cache hit fraction must be logged.
    writes=720 if eligible else 0
    hits=max(0,attempts-writes) if eligible else 0
    write_multiplier=m['cache_write_multiplier']
    # For OpenAI only 1,408 of 1,500 identical tokens lie on a 128-token boundary.
    cached_prefix=1408 if m['developer']=='OpenAI' else prefix
    reduction=hits*cached_prefix*(a-(c or 0))/1e6
    premium=writes*cached_prefix*a*(write_multiplier-1)/1e6
    storage=1500*720/1e6 if eligible and m['developer']=='Google' else 0
    cached=raw-reduction+premium+storage
    reasoning_unknown=m['model'] in ['Grok 4.7','gpt-oss-120b']
    return {'model':m['model'],'rate_evidence':m['evidence_status'],'uncached_usd':raw,
            'cache_scenario_usd':cached,'eligible_prefix':eligible,'billable_cached_prefix':cached_prefix if eligible else 0,
            'cache_creations_assumed':writes,'cache_hits_assumed':hits,'write_premium_usd':premium,
            'storage_usd':storage,'within_budget_uncached':None if reasoning_unknown and raw<=x['B'] else raw<=x['B'],
            'within_budget_cache_scenario':None if reasoning_unknown and cached<=x['B'] else cached<=x['B'],
            'reasoning_surcharge_unknown':reasoning_unknown}

COSTS=[model_cost(m,x) for m in MODELS]
for r in COSTS:
    print(r["model"],round(r["uncached_usd"],2),round(r["cache_scenario_usd"],2),r["within_budget_uncached"],r["within_budget_cache_scenario"])

GPT-4.1 mini 1512.09 1241.58 False False
GPT-4.1 nano 378.02 310.39 True True
Claude Haiku 4.5 4025.55 4025.55 False False
Claude Sonnet 4.5 12076.65 9483.76 False False
Gemini 2.5 Flash-Lite 378.02 292.65 True True
Grok 4.7 7069.8 7069.8 False False
Qwen3-235B-A22B-Instruct-2507 309.47 309.47 True True
Qwen3-32B 234.08 234.08 True True
Llama-3.3-70B-Instruct 358.4 358.4 True True
Mistral-Small-3.2-24B-Instruct 207.19 207.19 True True
DeepSeek-V3.1 932.79 932.79 True True
gpt-oss-120b 120.77 120.77 None None


## Latency sensitivity
TTFT grows by 0.00005 seconds per additional input token. 120 and 180 tokens/s and the model speeds are planning assumptions; the tool median is measured on one date, not measurements. The balanced primary CSV now has 20 successful weather, OpenAQ, HI and KB calls each on 25 November 2026; d=.713524150s. OpenAQ observation values are stale and are not current advice. The second date remains pending.

In [4]:
def latency(x,ttft,tps,delay):
    terms=[]
    for s in x['steps']:
        # Context-sensitive TTFT here is an assumption, not a provider result.
        prefill=ttft+.00005*(s['input']-2000)
        seconds=prefill+s['output']/tps
        terms.append({**s,'ttft_assumed_seconds':prefill,'tps_assumed':tps,'llm_seconds_expected':seconds*s['attempts_expected']})
    return {'terms':terms,'seconds':sum(t['llm_seconds_expected'] for t in terms)+(x['N']-1)*delay,
            'tool_delay_assumed':delay,'status':'SCENARIO, NOT MEASURED'}

print(json.dumps(latency(x,.3,120,.7135241500218399),indent=2))
print(json.dumps(latency(x,.2,180,.7135241500218399),indent=2))

{
  "terms": [
    {
      "k": 1,
      "input": 2000,
      "output": 350,
      "attempts_expected": 1.11,
      "ttft_assumed_seconds": 0.3,
      "tps_assumed": 120,
      "llm_seconds_expected": 3.5705
    },
    {
      "k": 2,
      "input": 2800,
      "output": 350,
      "attempts_expected": 1.11,
      "ttft_assumed_seconds": 0.33999999999999997,
      "tps_assumed": 120,
      "llm_seconds_expected": 3.6149
    },
    {
      "k": 3,
      "input": 3600,
      "output": 350,
      "attempts_expected": 1.11,
      "ttft_assumed_seconds": 0.38,
      "tps_assumed": 120,
      "llm_seconds_expected": 3.6593
    },
    {
      "k": 4,
      "input": 4400,
      "output": 350,
      "attempts_expected": 1.11,
      "ttft_assumed_seconds": 0.42,
      "tps_assumed": 120,
      "llm_seconds_expected": 3.7037
    },
    {
      "k": 5,
      "input": 5200,
      "output": 350,
      "attempts_expected": 1.11,
      "ttft_assumed_seconds": 0.45999999999999996,
      "tps_assumed": 

## Step-function break-even after the architectural change
One deterministic router, parallel AQ/weather tools, local HI/KB, one 120-token answer with 2,500 input tokens. Expected 1.11 attempts. Assumed 4-second self-host occupancy, the calculated 28 memory slots/GPU, A100 80GB advertised at $1.59/h. This is a capacity scenario, not a throughput certificate.

In [5]:
rate=(2500*(1+x["p"])*.10+120*(1+x["p"])*.40)/1e6
monthly_per_Q=(30+4*x["H"])*rate
slot_factor=(5*.35/120)*4/60
span=x["memory"]["concurrent_sequences_per_gpu"]/slot_factor
lease=720*1.59
print("API/query:",rate,"Monthly API at Q=1550:",x["Q"]*monthly_per_Q)
print("Q per GPU step:",span,"One GPU per month:",lease)
print("No crossing at any volume:",lease/span >= monthly_per_Q)
print("Memory capacity ignores compute/queueing limits; rerun with measured holding time.")

API/query: 0.0003307800000000001 Monthly API at Q=1550: 27.68628600000001
Q per GPU step: 28800.0 One GPU per month: 1144.8
No crossing at any volume: True
Memory capacity ignores compute/queueing limits; rerun with measured holding time.


## Evidence and limitations
See research/model_catalog.csv and evidence/tool_latency_raw.csv. A1500-token prefix is below the assumed 4096-token Haiku minimum; no cache reduction is taken there. For OpenAI, 1408 tokens lie on the 128-token cache boundary. Minimum sizes/lifetimes need current documentation revalidation. The screenshot and all twelve benchmark pairs are included. Some rate-limit/cache/speed fields and the second timing date remain unresolved. Preserve the distinction between old observation values and present API response timings.